## Step 1: Train-test split

Split the dataset before fitting any cleaning, imputation, outlier handling, encoding, scaling, or feature engineering rules. This prevents test-data leakage.


In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv(r"C:\Users\AYUSH PAWSHE\Desktop\CreditCard\data\credit_risk_dataset.csv")

In [4]:
from sklearn.model_selection import train_test_split

TARGET = "loan_status"

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("\nTraining target distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).rename("proportion"))


Training rows: 26064
Test rows: 6517

Training target distribution:
loan_status
0    20378
1     5686
Name: count, dtype: int64
loan_status
0    0.781845
1    0.218155
Name: proportion, dtype: float64

Test target distribution:
loan_status
0    0.781801
1    0.218199
Name: proportion, dtype: float64


## Step 2: Clean missing values and extreme outliers

Learn cleaning values from the training set only, then apply the same rules to both training and test data.


In [5]:
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Domain-clean impossible values before calculating imputation values.
def mark_impossible_values_as_missing(data):
    data = data.copy()
    data.loc[data["person_age"] > 100, "person_age"] = np.nan
    data.loc[data["person_emp_length"] > 60, "person_emp_length"] = np.nan
    data.loc[data["person_emp_length"] > (data["person_age"] - 14), "person_emp_length"] = np.nan
    return data

X_train_clean = mark_impossible_values_as_missing(X_train_clean)
X_test_clean = mark_impossible_values_as_missing(X_test_clean)

numeric_columns = X_train_clean.select_dtypes(include="number").columns

train_medians = X_train_clean[numeric_columns].median()
train_upper_caps = X_train_clean[numeric_columns].quantile(0.995)

def apply_numeric_cleaning(data):
    data = data.copy()
    data[numeric_columns] = data[numeric_columns].fillna(train_medians)
    data[numeric_columns] = data[numeric_columns].clip(upper=train_upper_caps, axis=1)
    return data

X_train_clean = apply_numeric_cleaning(X_train_clean)
X_test_clean = apply_numeric_cleaning(X_test_clean)

print("Remaining missing values in training set:", X_train_clean.isna().sum().sum())
print("Remaining missing values in test set:", X_test_clean.isna().sum().sum())
print("\nTraining upper caps learned from X_train only:")
print(train_upper_caps)


Remaining missing values in training set: 0
Remaining missing values in test set: 0

Training upper caps learned from X_train only:
person_age                        54.00
person_income                 300000.00
person_emp_length                 19.00
loan_amnt                      35000.00
loan_int_rate                     19.29
loan_percent_income                0.53
cb_person_cred_hist_length        23.00
Name: 0.995, dtype: float64


## Step 3: Engineer financial ratios

Create affordability and stability ratios from the cleaned train/test data. The dataset does not include existing monthly debt payments, so `dti_proxy` is an approximation based on requested loan amount divided by annual income.


In [6]:
def add_financial_ratios(data):
    data = data.copy()
    safe_income = data["person_income"].replace(0, np.nan)
    safe_age = data["person_age"].replace(0, np.nan)
    safe_credit_history = data["cb_person_cred_hist_length"].replace(0, np.nan)

    data["dti_proxy"] = data["loan_amnt"] / safe_income
    data["interest_burden"] = (data["loan_amnt"] * data["loan_int_rate"] / 100) / safe_income
    data["income_to_loan_ratio"] = safe_income / data["loan_amnt"].replace(0, np.nan)
    data["employment_to_age_ratio"] = data["person_emp_length"] / safe_age
    data["credit_history_to_age_ratio"] = data["cb_person_cred_hist_length"] / safe_age
    data["income_per_credit_year"] = safe_income / safe_credit_history

    ratio_columns = [
        "dti_proxy",
        "interest_burden",
        "income_to_loan_ratio",
        "employment_to_age_ratio",
        "credit_history_to_age_ratio",
        "income_per_credit_year",
    ]
    data[ratio_columns] = data[ratio_columns].replace([np.inf, -np.inf], np.nan)
    return data

X_train_features = add_financial_ratios(X_train_clean)
X_test_features = add_financial_ratios(X_test_clean)

new_ratio_columns = [col for col in X_train_features.columns if col not in X_train_clean.columns]
train_ratio_medians = X_train_features[new_ratio_columns].median()
X_train_features[new_ratio_columns] = X_train_features[new_ratio_columns].fillna(train_ratio_medians)
X_test_features[new_ratio_columns] = X_test_features[new_ratio_columns].fillna(train_ratio_medians)

print("New financial ratio features:")
print(new_ratio_columns)
print("\nTraining shape before ratios:", X_train_clean.shape)
print("Training shape after ratios:", X_train_features.shape)
print("\nSample engineered ratios:")
display(X_train_features[new_ratio_columns].head())


New financial ratio features:
['dti_proxy', 'interest_burden', 'income_to_loan_ratio', 'employment_to_age_ratio', 'credit_history_to_age_ratio', 'income_per_credit_year']

Training shape before ratios: (26064, 11)
Training shape after ratios: (26064, 17)

Sample engineered ratios:


,dti_proxy,interest_burden,income_to_loan_ratio,employment_to_age_ratio,credit_history_to_age_ratio,income_per_credit_year
15884,0.066150,0.004664,15.117188,0.160000,0.160000,60468.75
15138,0.083333,0.010150,12.000000,0.238095,0.190476,4500.00
7474,0.301887,0.037826,3.312500,0.400000,0.080000,26500.00
18212,0.297619,0.041607,3.360000,0.142857,0.285714,2100.00
6493,0.200000,0.015800,5.000000,0.080000,0.080000,25000.00


## Step 4: Build model-ready preprocessing

Use a `ColumnTransformer` to scale numeric columns and one-hot encode categorical columns. The transformer is fit on training data only, then reused on the test data.


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train_features.select_dtypes(include="number").columns.tolist()
categorical_features = X_train_features.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train_prepared = preprocessor.fit_transform(X_train_features)
X_test_prepared = preprocessor.transform(X_test_features)

encoded_feature_names = preprocessor.get_feature_names_out()

print("Numeric features:", len(numeric_features))
print(numeric_features)
print("\nCategorical features:", len(categorical_features))
print(categorical_features)
print("\nPrepared training shape:", X_train_prepared.shape)
print("Prepared test shape:", X_test_prepared.shape)
print("\nFirst 10 encoded feature names:")
print(encoded_feature_names[:10])


Numeric features: 13
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'dti_proxy', 'interest_burden', 'income_to_loan_ratio', 'employment_to_age_ratio', 'credit_history_to_age_ratio', 'income_per_credit_year']

Categorical features: 4
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

Prepared training shape: (26064, 32)
Prepared test shape: (6517, 32)

First 10 encoded feature names:
['numeric__person_age' 'numeric__person_income'
 'numeric__person_emp_length' 'numeric__loan_amnt'
 'numeric__loan_int_rate' 'numeric__loan_percent_income'
 'numeric__cb_person_cred_hist_length' 'numeric__dti_proxy'
 'numeric__interest_burden' 'numeric__income_to_loan_ratio']


## Step 5: Train and benchmark models

Train Logistic Regression, Random Forest, XGBoost, and LightGBM. Because the target is imbalanced, compare models mainly with PR-AUC and ROC-AUC instead of accuracy alone.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
        verbose=-1,
    ),
}

benchmark_rows = []
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train_prepared, y_train)
    y_proba = model.predict_proba(X_test_prepared)[:, 1]
    y_pred = (y_proba >= 0.50).astype(int)

    benchmark_rows.append({
        "model": model_name,
        "PR-AUC": average_precision_score(y_test, y_proba),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
    })
    trained_models[model_name] = model

benchmark_results = (
    pd.DataFrame(benchmark_rows)
    .sort_values(by="PR-AUC", ascending=False)
    .reset_index(drop=True)
)

best_model_name = benchmark_results.loc[0, "model"]
best_model = trained_models[best_model_name]

print("Class imbalance scale_pos_weight:", round(scale_pos_weight, 2))
print("Best model by PR-AUC:", best_model_name)
display(benchmark_results)


## Step 6: Generate SHAP rejection reasons

Use SHAP values from the best model to identify which features most increased a high-risk prediction, then convert those drivers into concise regulatory-style rejection reason candidates. In a production lending system, these reason codes should be reviewed for fair-lending and adverse-action compliance before use.


In [ ]:
import shap

def prepared_to_dataframe(prepared_matrix, feature_names):
    if hasattr(prepared_matrix, "toarray"):
        prepared_matrix = prepared_matrix.toarray()
    return pd.DataFrame(prepared_matrix, columns=feature_names)

X_test_explain = prepared_to_dataframe(X_test_prepared, encoded_feature_names)

explainer = shap.TreeExplainer(best_model)
raw_shap_values = explainer.shap_values(X_test_explain)

if isinstance(raw_shap_values, list):
    shap_values_for_risk = raw_shap_values[1]
elif getattr(raw_shap_values, "ndim", 0) == 3:
    shap_values_for_risk = raw_shap_values[:, :, 1]
else:
    shap_values_for_risk = raw_shap_values

def rejection_reason_from_feature(feature_name):
    clean_name = feature_name.replace("numeric__", "").replace("categorical__", "")

    reason_map = {
        "dti_proxy": "Requested loan amount is high relative to stated income.",
        "loan_percent_income": "Loan payment burden is high relative to stated income.",
        "interest_burden": "Estimated interest burden is high relative to stated income.",
        "income_to_loan_ratio": "Income support for the requested loan amount is limited.",
        "person_income": "Stated income is a material risk factor for this application.",
        "loan_amnt": "Requested loan amount is a material risk factor for this application.",
        "loan_int_rate": "Quoted interest rate indicates elevated credit risk.",
        "person_emp_length": "Employment history length is a material risk factor.",
        "employment_to_age_ratio": "Employment stability is limited relative to applicant age.",
        "cb_person_cred_hist_length": "Credit history length is a material risk factor.",
        "credit_history_to_age_ratio": "Credit history depth is limited relative to applicant age.",
        "income_per_credit_year": "Income relative to credit history depth is a material risk factor.",
        "person_age": "Applicant age profile is a material model risk factor.",
    }

    if clean_name.startswith("loan_grade_"):
        return "Loan grade indicates elevated credit risk."
    if clean_name.startswith("cb_person_default_on_file_Y"):
        return "Prior default history is present in the credit file."
    if clean_name.startswith("person_home_ownership_RENT"):
        return "Housing status indicates limited collateral or ownership stability."
    if clean_name.startswith("person_home_ownership_OTHER"):
        return "Housing status is a material risk factor."
    if clean_name.startswith("loan_intent_"):
        return "Loan purpose is associated with elevated repayment risk."

    return reason_map.get(clean_name, f"{clean_name} is a material risk factor.")

def top_three_rejection_reasons(applicant_position):
    applicant_shap = pd.Series(
        shap_values_for_risk[applicant_position],
        index=encoded_feature_names,
    ).sort_values(ascending=False)

    reasons = []
    for feature_name, shap_value in applicant_shap.items():
        if shap_value <= 0:
            continue
        reason = rejection_reason_from_feature(feature_name)
        if reason not in reasons:
            reasons.append(reason)
        if len(reasons) == 3:
            break
    return reasons

test_risk_scores = best_model.predict_proba(X_test_prepared)[:, 1]
high_risk_positions = np.where(test_risk_scores >= 0.50)[0]

reason_examples = []
for position in high_risk_positions[:5]:
    reason_examples.append({
        "test_position": int(position),
        "predicted_default_risk": test_risk_scores[position],
        "top_3_rejection_reasons": top_three_rejection_reasons(position),
    })

display(pd.DataFrame(reason_examples))
